In [1]:
import json
import psycopg
from psycopg.types.json import Json

In [2]:
conn = psycopg.connect(
    dbname = "leetlog",
    user="postgres",
    password="root",
    host="localhost"
)

In [3]:
cur = conn.cursor()

In [4]:
cur.execute("SELECT * FROM StarterCode")
cur.fetchall()

[(11,
  11,
  'cpp',
  'class Solution {\npublic:\n    int hIndex(vector<int>& citations) {\n        \n    }\n};',
  '#include <bits/stdc++.h>\nusing namespace std;\n\nint main() {\n    int citations_n; cin >> citations_n;\n    vector<int> citations(citations_n);\n    for (int i = 0; i < citations_n; i++) cin >> citations[i];\n    \n    Solution obj;\n    auto ans = obj.hIndex(citations);\n    cout << ans;\n}'),
 (12,
  12,
  'cpp',
  'class Solution {\npublic:\n    int findRadius(vector<int>& houses, vector<int>& heaters) {\n        \n    }\n};',
  '#include <bits/stdc++.h>\nusing namespace std;\n\nint main() {\n    int houses_n; cin >> houses_n;\n    vector<int> houses(houses_n);\n    for (int i = 0; i < houses_n; i++) cin >> houses[i];\n    int heaters_n; cin >> heaters_n;\n    vector<int> heaters(heaters_n);\n    for (int i = 0; i < heaters_n; i++) cin >> heaters[i];\n    \n    Solution obj;\n    auto ans = obj.findRadius(houses, heaters);\n    cout << ans;\n}'),
 (13,
  13,
  'cpp

#### Fetch  data from 3 json files


In [19]:
with open("../starter_code.json","r") as file:
    starterCode = json.load(file)
with open("../dsa_descriptions.json","r",encoding="utf-8") as file:
    description = json.load(file)

with open("../dsa_data.json","r",encoding="utf-8") as file:
    data = json.load(file)

In [20]:
print(len(starterCode))
print(len(description))

174
174


### Find Missing Problems
before touching DB , we need to ensure the 3 file are synced together so no problem is misses out


In [21]:
## Problems missing from StarterCode
missing = set(description) - set(starterCode)

print(len(missing))
for p in sorted(missing):
    print(p)

0


In [22]:
## Problems that exist only in StarterCode
extra = set(starterCode) - set(description)

print(len(extra))
for p in sorted(extra):
    print(p)

0


#### Store them in a Seperate json file and get the StarterCode from AI

In [23]:
missing = set(description) - set(starterCode)
missing_editorial = {}
problems_needing_starter_code = {}
for problem in missing:
    problems_needing_starter_code[problem] = description[problem]

for topic,problem in data.items():
    for problem_name,code in problem.items():
        if problem_name in missing:
            if topic not in missing_editorial:
                missing_editorial[topic]={}
            missing_editorial[topic][problem_name] = code





In [24]:
with open("missing_editorial.json","w",encoding="utf-8") as file:
    json.dump(missing_editorial,file,indent=2,ensure_ascii=False)
with open("problems_needing_starter_code.json","w",encoding="utf-8") as file:
    json.dump(problems_needing_starter_code,file,indent=2,ensure_ascii=False)

In [25]:
print(len(problems_needing_starter_code))
print(len(missing_editorial))


0
0


### Update the original Starter Code file with the new ones

In [12]:
with open("starter_code_for_missing_problems.json","r",encoding="utf-8") as file:
    new_starter_codes = json.load(file)
print(new_starter_codes)

{'Reverse String.cpp': {'language': 'cpp', 'template_code': 'class Solution {\npublic:\n    void reverseString(vector<char>& s) {\n        \n    }\n};', 'wrapper_main': '#include <bits/stdc++.h>\nusing namespace std;\n\nint main() {\n    int s_n; cin >> s_n;\n    vector<char> s(s_n);\n    for (int i = 0; i < s_n; i++) cin >> s[i];\n    \n    Solution obj;\n    obj.reverseString(s);\n    for (size_t i = 0; i < s.size(); i++) cout << s[i] << " ";\n    cout << "\\n";\n}'}}


In [13]:
starterCode.update(new_starter_codes)

In [14]:
with open("../starter_code.json","w",encoding="utf-8") as file:
    json.dump(starterCode,file,indent=2,ensure_ascii=False)

#### Inserting into DB


In [15]:
problem_map = {}
cur.execute("SELECT problem_key,id FROM Problems")

for problem_key,id in cur.fetchall():
    problem_map[problem_key] = id
print(problem_map)

{'Median in a row-wise sorted Matrix': 19, 'Nim game': 23, 'Allocate-Minimum-Pages': 2, 'Plus-One': 169, 'Remove Nodes From Linked List': 50, 'Remove-Duplicates-from-Sorted-Array': 170, 'Remove-Element': 171, 'Previous Smaller Element': 105, 'Count of Smaller Numbers After Self': 3, 'Find Right Interval': 4, 'Aggressive-cows': 1, 'Find Smallest Letter Greater Than Target': 5, 'Find a Peak Element II': 6, 'Find the Duplicate Number': 7, 'Find-Minimum-in-Rotated-Sorted-Array-II': 8, 'First and Last Occurrences': 9, 'First-Bad-Version': 10, 'H-Index II': 11, 'Heaters': 12, 'Intersection-of-Two-Arrays': 13, 'K-diff Pairs in an Array': 14, 'K-th element of two Arrays': 15, 'Kth Smallest Element in a Sorted Matrix': 16, 'Kth Smallest Product of Two Sorted Arrays': 17, 'Kth-Missing-Positive-Number': 18, 'Median-of-Two-Sorted-Arrays': 20, 'Minimize Max Distance to Gas Station': 21, 'Number of occurrence': 24, 'Row With Maximum Ones': 25, 'Search a 2D Matrix': 26, 'Search in Rotated Sorted Arra

In [16]:
with open("../starter_code.json","r") as file:
    starterCode = json.load(file)

upsert_query = """ 
                INSERT INTO StarterCode (
                problem_id,
                language,
                template_code,
                wrapper_main
                )
                values(%s,%s,%s,%s)

                ON CONFLICT(problem_id,language)
                DO UPDATE 
                    SET
                        template_code = EXCLUDED.template_code,
                        wrapper_main = EXCLUDED.wrapper_main

"""
cnt = 0
for problem,meta in starterCode.items():
    language = meta["language"]
    template_code = meta["template_code"]
    wrapper_main = meta["wrapper_main"]
    problem_id = problem_map[problem.removesuffix(".cpp")]
    values  = (problem_id,language,template_code,wrapper_main)
    try:
        cur.execute(upsert_query, values)
        print(f"✅ Upserted: {problem.removesuffix('.cpp')}")
        cnt = cnt+1
    except Exception as e:
        conn.rollback()
        print(f"❌ Failed: {problem.removesuffix('.cpp')}")
        print(e)
        break
conn.commit()
print("--------------------------------")
print("StarterCode import completed!")
print(f"Total Upserts: {cnt}")
print("--------------------------------")
 
    

✅ Upserted: Aggressive-cows
✅ Upserted: Allocate-Minimum-Pages
✅ Upserted: Count of Smaller Numbers After Self
✅ Upserted: Find Right Interval
✅ Upserted: Find Smallest Letter Greater Than Target
✅ Upserted: Find a Peak Element II
✅ Upserted: Find the Duplicate Number
✅ Upserted: Find-Minimum-in-Rotated-Sorted-Array-II
✅ Upserted: First and Last Occurrences
✅ Upserted: First-Bad-Version
✅ Upserted: H-Index II
✅ Upserted: Heaters
✅ Upserted: Intersection-of-Two-Arrays
✅ Upserted: K-diff Pairs in an Array
✅ Upserted: K-th element of two Arrays
✅ Upserted: Kth Smallest Element in a Sorted Matrix
✅ Upserted: Kth Smallest Product of Two Sorted Arrays
✅ Upserted: Kth-Missing-Positive-Number
✅ Upserted: Median in a row-wise sorted Matrix
✅ Upserted: Median-of-Two-Sorted-Arrays
✅ Upserted: Minimize Max Distance to Gas Station
✅ Upserted: Minimum Size Subarray Sum
✅ Upserted: Nim game
✅ Upserted: Number of occurrence
✅ Upserted: Row With Maximum Ones
✅ Upserted: Search a 2D Matrix
✅ Upserted: S

In [17]:
# Find problems without starter code

cur.execute("""SELECT p.problem_key
FROM Problems p
LEFT JOIN StarterCode s
ON p.id = s.problem_id
WHERE s.problem_id IS NULL;""").fetchall()

[]

In [18]:
# Find orphan starter codes
cur.execute("""SELECT s.*
FROM StarterCode s
LEFT JOIN Problems p
ON s.problem_id = p.id
WHERE p.id IS NULL;""").fetchall()

[]